# 3.3 - Tree-Based Models: Random Forest, XGBoost, LightGBM

**Taller de Programación - UBA FCE | Grupo JLP**

---

## Objetivo

Implementar modelos basados en árboles para capturar **relaciones no-lineales** y **interacciones complejas**:

1. **Random Forest:** Ensemble de árboles decorrelacionados via bagging
2. **XGBoost:** Gradient boosting con regularización y velocidad
3. **LightGBM:** Gradient boosting más rápido y eficiente en memoria

**Ventajas sobre modelos lineales:**
- Capturan no-linealidades sin feature engineering
- Robusto a outliers y escala de variables
- Feature importance intrínseca
- Manejan interacciones automáticamente

**Optimización:** Grid search con TimeSeriesSplit para hiperparámetros óptimos.

## Setup

In [11]:
# Imports
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import lightgbm as lgb
from tqdm.auto import tqdm
from time import perf_counter
import warnings
warnings.filterwarnings('ignore')

# Agregar src al path
BASE_DIR = Path.cwd().parents[1]
sys.path.append(str(BASE_DIR / 'src'))

from config import PROCESSED_DIR, logger

# Configurar pandas display
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

# Definir commodities target
TARGET_COMMODITIES = ['Corn', 'Soybeans', 'Wheat']

print(f"✓ Base directory: {BASE_DIR}")
print(f"✓ Target commodities: {', '.join(TARGET_COMMODITIES)}")
print(f"\nVersiones:")
print(f"  XGBoost: {xgb.__version__}")
print(f"  LightGBM: {lgb.__version__}")

✓ Base directory: c:\Users\AdministradorIT\OneDrive\UBA\Taller de Programacion\TPFinal
✓ Target commodities: Corn, Soybeans, Wheat

Versiones:
  XGBoost: 3.0.1
  LightGBM: 4.6.0


## 1. Cargar Features y Split Temporal

In [6]:
# Cargar dataset
input_file = PROCESSED_DIR / 'features_selected_modeling.csv'

if not input_file.exists():
    raise FileNotFoundError(f"No se encontró {input_file}. Ejecuta notebook 3.1 primero.")

df = pd.read_csv(input_file, parse_dates=['date'])

print(f"✓ Dataset cargado: {input_file.name}")
print(f"  Dimensiones: {df.shape}")

# Separar features y targets
target_cols = [f'{c}_target_t7' for c in TARGET_COMMODITIES]
feature_cols = [c for c in df.columns if c not in ['date'] + target_cols]

# Split temporal (igual que notebook 3.2)
split_date = '2023-01-01'
train_idx = df['date'] < split_date
test_idx = df['date'] >= split_date

X_train = df.loc[train_idx, feature_cols]
X_test = df.loc[test_idx, feature_cols]
y_train = df.loc[train_idx, target_cols]
y_test = df.loc[test_idx, target_cols]

print(f"\nTrain/Test split:")
print(f"  Train: {X_train.shape[0]:,} obs")
print(f"  Test: {X_test.shape[0]:,} obs")
print(f"  Features: {len(feature_cols)}")

# CRÍTICO: Limpiar NaN en targets (filas sin target válido)
print(f"\nVerificando NaN en targets...")
nan_train = y_train.isnull().sum().sum()
nan_test = y_test.isnull().sum().sum()
print(f"  NaN en y_train: {nan_train} | NaN en y_test: {nan_test}")

if nan_train > 0 or nan_test > 0:
    print(f"  Eliminando filas con NaN en targets...")
    # Identificar filas válidas (todas las columnas target sin NaN)
    valid_train = y_train.notna().all(axis=1)
    valid_test = y_test.notna().all(axis=1)
    
    X_train = X_train[valid_train]
    y_train = y_train[valid_train]
    X_test = X_test[valid_test]
    y_test = y_test[valid_test]
    
    print(f"  Filas eliminadas: train={(~valid_train).sum()} | test={(~valid_test).sum()}")
    print(f"  Nuevo tamaño: train={X_train.shape[0]:,} | test={X_test.shape[0]:,}")

# Verificación final
assert y_train.isnull().sum().sum() == 0, "ERROR: Quedan NaN en y_train"
assert y_test.isnull().sum().sum() == 0, "ERROR: Quedan NaN en y_test"
print(f"✓ Targets limpios sin NaN")

# Limpiar NaN/inf en features (tree models los pueden manejar pero mejor limpiar)
print(f"\nVerificando NaN/inf en features...")
inf_train = np.isinf(X_train).sum().sum()
inf_test = np.isinf(X_test).sum().sum()
nan_train_feat = X_train.isnull().sum().sum()
nan_test_feat = X_test.isnull().sum().sum()
print(f"  Inf: train={inf_train} | test={inf_test}")
print(f"  NaN: train={nan_train_feat} | test={nan_test_feat}")

if inf_train > 0 or inf_test > 0:
    print("  Reemplazando infinitos por NaN...")
    X_train = X_train.replace([np.inf, -np.inf], np.nan)
    X_test = X_test.replace([np.inf, -np.inf], np.nan)

if X_train.isnull().sum().sum() > 0 or X_test.isnull().sum().sum() > 0:
    print("  Aplicando median imputation (usando mediana de train)...")
    medians = X_train.median()
    X_train = X_train.fillna(medians)
    X_test = X_test.fillna(medians)
    print(f"✓ Features limpios sin NaN/inf")

✓ Dataset cargado: features_selected_modeling.csv
  Dimensiones: (6724, 40)

Train/Test split:
  Train: 5,987 obs
  Test: 737 obs
  Features: 36

Verificando NaN en targets...
  NaN en y_train: 1088 | NaN en y_test: 75
  Eliminando filas con NaN en targets...
  Filas eliminadas: train=424 | test=25
  Nuevo tamaño: train=5,563 | test=712
✓ Targets limpios sin NaN

Verificando NaN/inf en features...
  Inf: train=0 | test=0
  NaN: train=3086 | test=126
  Aplicando median imputation (usando mediana de train)...
✓ Features limpios sin NaN/inf


## 2. Función de Evaluación (reutilizada de 3.2)

In [7]:
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    """
    Evalúa modelo de regresión con métricas estándar
    """
    results = {}
    
    for target_col in y_train.columns:
        commodity = target_col.replace('_target_t7', '')
        
        # Predictions
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)
        
        # Metrics
        train_rmse = np.sqrt(mean_squared_error(y_train[target_col], y_train_pred))
        test_rmse = np.sqrt(mean_squared_error(y_test[target_col], y_test_pred))
        train_mae = mean_absolute_error(y_train[target_col], y_train_pred)
        test_mae = mean_absolute_error(y_test[target_col], y_test_pred)
        train_r2 = r2_score(y_train[target_col], y_train_pred)
        test_r2 = r2_score(y_test[target_col], y_test_pred)
        
        # Directional accuracy
        y_train_direction = np.sign(y_train[target_col] - y_train[target_col].shift(1))
        y_train_pred_direction = np.sign(y_train_pred - y_train[target_col].shift(1))
        train_dir_acc = (y_train_direction == y_train_pred_direction).mean()
        
        y_test_direction = np.sign(y_test[target_col] - y_test[target_col].shift(1))
        y_test_pred_direction = np.sign(y_test_pred - y_test[target_col].shift(1))
        test_dir_acc = (y_test_direction == y_test_pred_direction).mean()
        
        results[commodity] = {
            'train_rmse': train_rmse,
            'test_rmse': test_rmse,
            'train_mae': train_mae,
            'test_mae': test_mae,
            'train_r2': train_r2,
            'test_r2': test_r2,
            'train_dir_acc': train_dir_acc,
            'test_dir_acc': test_dir_acc,
            'overfitting_gap': train_r2 - test_r2
        }
    
    return results

print("✓ Función de evaluación definida")

✓ Función de evaluación definida


---

## MODELO 1: Random Forest

**Algoritmo:** Bagging de árboles decorrelacionados.

**Hiperparámetros clave:**
- `n_estimators`: Número de árboles (más árboles = mejor pero más lento)
- `max_depth`: Profundidad máxima de árboles (controla overfitting)
- `min_samples_split`: Mínimo de muestras para split (mayor = más generalización)
- `max_features`: Features a considerar en cada split ('sqrt' típico)

**Ventajas:**
- Robusto a overfitting (promedio de árboles)
- Feature importance confiable
- Pocas asunciones sobre datos

In [8]:
# Grid de hiperparámetros para Random Forest
rf_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2']
}

rf_models = {}
rf_results = {}
rf_best_params = {}
rf_feature_importance = {}

print(f"\n{'='*80}")
print(f"RANDOM FOREST")
print(f"{'='*80}")

with tqdm(target_cols, desc="RF Models", unit="commodity") as pbar:
    for target_col in pbar:
        commodity = target_col.replace('_target_t7', '')
        pbar.set_description(f"RF: {commodity}")
        
        start_time = perf_counter()
        print(f"\n--- {commodity} ---")
        print(f"  Iniciando Grid Search ({len(rf_param_grid['n_estimators']) * len(rf_param_grid['max_depth']) * len(rf_param_grid['min_samples_split']) * len(rf_param_grid['max_features'])} combinaciones)...")
        
        # Grid search con TimeSeriesSplit
        tscv = TimeSeriesSplit(n_splits=5)
        rf = RandomForestRegressor(random_state=42, n_jobs=-1)
        
        grid_search = GridSearchCV(
            rf,
            rf_param_grid,
            cv=tscv,
            scoring='neg_root_mean_squared_error',
            n_jobs=-1,
            verbose=2  # Mostrar progreso de GridSearchCV
        )
        
        grid_search.fit(X_train, y_train[target_col])
        elapsed = perf_counter() - start_time
    
    # Mejores parámetros
    best_params = grid_search.best_params_
    rf_best_params[commodity] = best_params
    print(f"  Mejores hiperparámetros:")
    for k, v in best_params.items():
        print(f"    {k}: {v}")
    
    # Entrenar modelo con mejores parámetros
    rf_best = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
    rf_best.fit(X_train, y_train[target_col])
    rf_models[commodity] = rf_best
    
    # Feature importance
    feature_imp = pd.DataFrame({
        'feature': feature_cols,
        'importance': rf_best.feature_importances_
    }).sort_values('importance', ascending=False)
    rf_feature_importance[commodity] = feature_imp
    
    print(f"\n  Top 10 features más importantes:")
    for idx, row in feature_imp.head(10).iterrows():
        print(f"    {row['feature']:50s}: {row['importance']:.6f}")
    
    # Evaluar
        results = evaluate_model(rf_best, X_train, X_test,
                                 y_train[[target_col]], y_test[[target_col]], 'Random Forest')
        rf_results[commodity] = results[commodity]
        
        # Imprimir resultados
        r = results[commodity]
        print(f"\n  Resultados:")
        print(f"    Train RMSE: {r['train_rmse']:.4f} | Test RMSE: {r['test_rmse']:.4f}")
        print(f"    Train R²:   {r['train_r2']:.4f} | Test R²:   {r['test_r2']:.4f}")
        print(f"    Test Dir Acc: {r['test_dir_acc']:.2%}")
        print(f"    Overfitting gap: {r['overfitting_gap']:.4f}")
        print(f"    Tiempo: {elapsed:.1f}s")
        
        # Actualizar postfix de tqdm
        pbar.set_postfix({'Test_R2': f"{r['test_r2']:.3f}", 'Time': f"{elapsed:.0f}s"})

print(f"\n{'='*80}")


RANDOM FOREST

--- Corn ---
  Iniciando Grid Search (72 combinaciones)...


  Mejores hiperparámetros:
    max_depth: 10
    max_features: sqrt
    min_samples_split: 10
    n_estimators: 100

  Top 10 features más importantes:
    Corn_lag3                                         : 0.146898
    Soybean_Oil_bb_lower7                             : 0.129762
    Soybeans_bb_lower7                                : 0.119381
    Soybean_Oil_lag1                                  : 0.099821
    Wheat_ma90                                        : 0.093036
    Soybean_Oil_ma90                                  : 0.086428
    Soybeans_ma7                                      : 0.057789
    Wheat_ma7                                         : 0.052009
    Wheat_bb_lower90                                  : 0.033082
    Gold_ma90                                         : 0.029845

  Top 10 features más importantes:
    Corn_lag3                                         : 0.146898
    Soybean_Oil_bb_lower7                             : 0.129762
    Soybeans_bb_lower7          

---

## MODELO 2: XGBoost

**Algoritmo:** Gradient boosting con regularización L1/L2.

**Diferencias vs Random Forest:**
- Boosting (secuencial) vs Bagging (paralelo)
- Cada árbol corrige errores del anterior
- Más propenso a overfitting pero mayor potencia

**Hiperparámetros clave:**
- `n_estimators`: Número de boosting rounds
- `learning_rate`: Velocidad de aprendizaje (típico 0.01-0.3)
- `max_depth`: Profundidad de árboles (menor que RF)
- `subsample`: Proporción de muestras por árbol (reduce overfitting)
- `colsample_bytree`: Proporción de features por árbol
- `reg_alpha` (L1), `reg_lambda` (L2): Regularización

In [ ]:
# Grid de hiperparámetros para XGBoost
xgb_param_grid = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb_models = {}
xgb_results = {}
xgb_best_params = {}

print(f"\n{'='*80}")
print(f"XGBOOST")
print(f"{'='*80}")

with tqdm(target_cols, desc="XGB Models", unit="commodity") as pbar:
    for target_col in pbar:
        commodity = target_col.replace('_target_t7', '')
        pbar.set_description(f"XGB: {commodity}")
        
        start_time = perf_counter()
        print(f"\n--- {commodity} ---")
        print(f"  Iniciando Grid Search ({len(xgb_param_grid['n_estimators']) * len(xgb_param_grid['learning_rate']) * len(xgb_param_grid['max_depth']) * len(xgb_param_grid['subsample']) * len(xgb_param_grid['colsample_bytree'])} combinaciones)...")
        
        # Grid search
        tscv = TimeSeriesSplit(n_splits=5)
        xgb_model = xgb.XGBRegressor(
            random_state=42,
            tree_method='hist',  # Más rápido
            n_jobs=-1
        )
        
        grid_search = GridSearchCV(
            xgb_model,
            xgb_param_grid,
            cv=tscv,
            scoring='neg_root_mean_squared_error',
            n_jobs=-1,
            verbose=2
        )
        
        grid_search.fit(X_train, y_train[target_col])
        elapsed = perf_counter() - start_time
    
        # Mejores parámetros
        best_params = grid_search.best_params_
        xgb_best_params[commodity] = best_params
        print(f"  Mejores hiperparámetros:")
        for k, v in best_params.items():
            print(f"    {k}: {v}")
        
        # Entrenar modelo con mejores parámetros
        xgb_best = xgb.XGBRegressor(
            **best_params,
            random_state=42,
            tree_method='hist',
            n_jobs=-1
        )
        xgb_best.fit(X_train, y_train[target_col])
        xgb_models[commodity] = xgb_best
        
        # Evaluar
        results = evaluate_model(xgb_best, X_train, X_test,
                                    y_train[[target_col]], y_test[[target_col]], 'XGBoost')
        xgb_results[commodity] = results[commodity]
            
        # Imprimir resultados
        r = results[commodity]
        print(f"\n  Resultados:")
        print(f"    Train RMSE: {r['train_rmse']:.4f} | Test RMSE: {r['test_rmse']:.4f}")
        print(f"    Train R²:   {r['train_r2']:.4f} | Test R²:   {r['test_r2']:.4f}")
        print(f"    Test Dir Acc: {r['test_dir_acc']:.2%}")
        print(f"    Overfitting gap: {r['overfitting_gap']:.4f}")
        print(f"    Tiempo: {elapsed:.1f}s")
            
        # Actualizar postfix de tqdm
        pbar.set_postfix({'Test_R2': f"{r['test_r2']:.3f}", 'Time': f"{elapsed:.0f}s"})

print(f"\n{'='*80}")


XGBOOST


XGB Models:   0%|          | 0/3 [00:00<?, ?commodity/s]


--- Corn ---
  Iniciando Grid Search (108 combinaciones)...
Fitting 5 folds for each of 108 candidates, totalling 540 fits

--- Soybeans ---
  Iniciando Grid Search (108 combinaciones)...
Fitting 5 folds for each of 108 candidates, totalling 540 fits

--- Wheat ---
  Iniciando Grid Search (108 combinaciones)...
Fitting 5 folds for each of 108 candidates, totalling 540 fits
  Mejores hiperparámetros:
    colsample_bytree: 0.8
    learning_rate: 0.01
    max_depth: 3
    n_estimators: 200
    subsample: 0.8

  Resultados:
    Train RMSE: 42.3557 | Test RMSE: 30.1871
    Train R²:   0.9487 | Test R²:   0.7747
    Test Dir Acc: 52.39%
    Overfitting gap: 0.1740
    Tiempo: 154.6s



---

## MODELO 3: LightGBM

**Algoritmo:** Gradient boosting con histogram-based learning.

**Ventajas vs XGBoost:**
- **Más rápido:** Crece árboles leaf-wise (no level-wise)
- **Menor memoria:** Discretiza features continuas
- **Mejor con datos grandes:** Optimizado para datasets masivos

**Hiperparámetros similares a XGBoost:**
- `num_leaves`: Número de hojas (leaf-wise growth)
- `learning_rate`, `n_estimators`, `subsample`, `colsample_bytree`
- `reg_alpha`, `reg_lambda`: Regularización

In [ ]:
# Grid de hiperparámetros para LightGBM
lgb_param_grid = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'num_leaves': [31, 63, 127],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

lgb_models = {}
lgb_results = {}
lgb_best_params = {}

print(f"\n{'='*80}")
print(f"LIGHTGBM")
print(f"{'='*80}")

with tqdm(target_cols, desc="LGB Models", unit="commodity") as pbar:
    for target_col in pbar:
        commodity = target_col.replace('_target_t7', '')
        pbar.set_description(f"LGB: {commodity}")
        
        start_time = perf_counter()
        print(f"\n--- {commodity} ---")
        print(f"  Iniciando Grid Search ({len(lgb_param_grid['n_estimators']) * len(lgb_param_grid['learning_rate']) * len(lgb_param_grid['num_leaves']) * len(lgb_param_grid['subsample']) * len(lgb_param_grid['colsample_bytree'])} combinaciones)...")
        
        # Grid search
        tscv = TimeSeriesSplit(n_splits=5)
        lgb_model = lgb.LGBMRegressor(
            random_state=42,
            n_jobs=-1,
            verbose=-1  # Silenciar warnings
        )
        
        grid_search = GridSearchCV(
            lgb_model,
            lgb_param_grid,
            cv=tscv,
            scoring='neg_root_mean_squared_error',
            n_jobs=-1,
            verbose=2
        )
        
        grid_search.fit(X_train, y_train[target_col])
        elapsed = perf_counter() - start_time
    
    # Mejores parámetros
    best_params = grid_search.best_params_
    lgb_best_params[commodity] = best_params
    print(f"  Mejores hiperparámetros:")
    for k, v in best_params.items():
        print(f"    {k}: {v}")
    
    # Entrenar modelo con mejores parámetros
    lgb_best = lgb.LGBMRegressor(
        **best_params,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
    lgb_best.fit(X_train, y_train[target_col])
    lgb_models[commodity] = lgb_best
    
    # Evaluar
    results = evaluate_model(lgb_best, X_train, X_test,y_train[[target_col]], y_test[[target_col]], 'LightGBM')
    lgb_results[commodity] = results[commodity]
        
    # Imprimir resultados
    r = results[commodity]
    print(f"\n  Resultados:")
    print(f"    Train RMSE: {r['train_rmse']:.4f} | Test RMSE: {r['test_rmse']:.4f}")
    print(f"    Train R²:   {r['train_r2']:.4f} | Test R²:   {r['test_r2']:.4f}")
    print(f"    Test Dir Acc: {r['test_dir_acc']:.2%}")
    print(f"    Overfitting gap: {r['overfitting_gap']:.4f}")
    print(f"    Tiempo: {elapsed:.1f}s")
        
    # Actualizar postfix de tqdm
    pbar.set_postfix({'Test_R2': f"{r['test_r2']:.3f}", 'Time': f"{elapsed:.0f}s"})

print(f"\n{'='*80}")


LIGHTGBM


LGB Models:   0%|          | 0/3 [00:00<?, ?commodity/s]


--- Corn ---
  Iniciando Grid Search (108 combinaciones)...
Fitting 5 folds for each of 108 candidates, totalling 540 fits


---

## 4. Comparación: Tree Models vs Baseline

In [ ]:
# Cargar resultados baseline (del notebook 3.2)
baseline_file = PROCESSED_DIR / 'baseline_models_results.json'

if baseline_file.exists():
    with open(baseline_file, 'r') as f:
        baseline_data = json.load(f)
    baseline_results_list = baseline_data['results']
else:
    baseline_results_list = []
    print("No se encontraron resultados baseline (notebook 3.2). Comparando solo tree models.")

# Consolidar resultados tree models
comparison_data = baseline_results_list.copy()

for commodity in TARGET_COMMODITIES:
    for model_name, results_dict in [('Random Forest', rf_results),
                                       ('XGBoost', xgb_results),
                                       ('LightGBM', lgb_results)]:
        if commodity in results_dict:
            r = results_dict[commodity]
            comparison_data.append({
                'Commodity': commodity,
                'Model': model_name,
                'Test RMSE': r['test_rmse'],
                'Test MAE': r['test_mae'],
                'Test R²': r['test_r2'],
                'Test Dir Acc': r['test_dir_acc'],
                'Overfitting Gap': r['overfitting_gap']
            })

comparison_df = pd.DataFrame(comparison_data)

print(f"\n{'='*80}")
print(f"COMPARACIÓN: TREE MODELS vs BASELINE")
print(f"{'='*80}\n")

# Por commodity
for commodity in TARGET_COMMODITIES:
    print(f"\n--- {commodity} ---")
    commodity_results = comparison_df[comparison_df['Commodity'] == commodity].copy()
    commodity_results = commodity_results.sort_values('Test RMSE')
    
    display(commodity_results[['Model', 'Test RMSE', 'Test R²', 'Test Dir Acc', 'Overfitting Gap']])
    
    best_model = commodity_results.iloc[0]['Model']
    best_rmse = commodity_results.iloc[0]['Test RMSE']
    print(f"\n✓ Mejor modelo: {best_model} (RMSE: {best_rmse:.4f})")

print(f"\n{'='*80}")

### Visualización: Test RMSE por Modelo

In [ ]:
# Plot comparativo
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, commodity in enumerate(TARGET_COMMODITIES):
    commodity_results = comparison_df[comparison_df['Commodity'] == commodity].sort_values('Test RMSE')
    
    ax = axes[idx]
    ax.barh(commodity_results['Model'], commodity_results['Test RMSE'], color='steelblue')
    ax.set_xlabel('Test RMSE', fontsize=12)
    ax.set_title(f'{commodity}', fontsize=14, fontweight='bold')
    ax.invert_yaxis()
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(BASE_DIR / 'reports' / 'figures' / 'tree_models_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Gráfico guardado: reports/figures/tree_models_comparison.png")

### Feature Importance: Random Forest

In [ ]:
# Plot feature importance para cada commodity
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for idx, commodity in enumerate(TARGET_COMMODITIES):
    feature_imp = rf_feature_importance[commodity].head(15)
    
    ax = axes[idx]
    ax.barh(range(len(feature_imp)), feature_imp['importance'], color='forestgreen')
    ax.set_yticks(range(len(feature_imp)))
    ax.set_yticklabels(feature_imp['feature'], fontsize=9)
    ax.set_xlabel('Importance (Gini)', fontsize=12)
    ax.set_title(f'{commodity} - Top 15 Features', fontsize=14, fontweight='bold')
    ax.invert_yaxis()
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(BASE_DIR / 'reports' / 'figures' / 'rf_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Gráfico guardado: reports/figures/rf_feature_importance.png")

---

## 5. Guardar Modelos y Resultados

In [ ]:
import pickle

# Guardar modelos tree-based
models_dir = BASE_DIR / 'models'
models_dir.mkdir(exist_ok=True)

all_models = {
    'random_forest': rf_models,
    'xgboost': xgb_models,
    'lightgbm': lgb_models,
    'feature_cols': feature_cols
}

models_file = models_dir / 'tree_models.pkl'
with open(models_file, 'wb') as f:
    pickle.dump(all_models, f)

print(f"✓ Modelos guardados: {models_file}")

# Guardar resultados en JSON
results_summary = {
    'fecha_generacion': pd.Timestamp.now().isoformat(),
    'models': ['Random Forest', 'XGBoost', 'LightGBM'],
    'commodities': TARGET_COMMODITIES,
    'split_date': split_date,
    'results': comparison_df.to_dict(orient='records'),
    'best_models': {},
    'hyperparameters': {
        'random_forest': rf_best_params,
        'xgboost': xgb_best_params,
        'lightgbm': lgb_best_params
    }
}

# Identificar mejor modelo por commodity
for commodity in TARGET_COMMODITIES:
    commodity_results = comparison_df[comparison_df['Commodity'] == commodity].sort_values('Test RMSE')
    results_summary['best_models'][commodity] = commodity_results.iloc[0]['Model']

results_file = PROCESSED_DIR / 'tree_models_results.json'
with open(results_file, 'w') as f:
    json.dump(results_summary, f, indent=2)

print(f"✓ Resultados guardados: {results_file}")

# Guardar feature importance de Random Forest
for commodity in TARGET_COMMODITIES:
    rf_feature_importance[commodity].to_csv(
        PROCESSED_DIR / f'rf_feature_importance_{commodity.lower()}.csv',
        index=False
    )

print(f"✓ Feature importance guardado (3 archivos CSV)")